In [1]:
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import joblib
import shap

# 定义与原始训练脚本相同的模型类结构
class MyModel(nn.Module):
    def __init__(self, input_dim=12):
        super(MyModel, self).__init__()
        self.layer1 = nn.Linear(input_dim, 450)
        self.bn1 = nn.BatchNorm1d(450)
        self.relu = nn.ReLU()
        self.layer2 = nn.Linear(450, 1)

    def forward(self, x):
        x = self.layer1(x)
        x = self.bn1(x)
        x = self.relu(x)
        x = self.layer2(x)
        return x

# ====================== 1. 定义路径 ======================
model_dir = r"D:\文成数据库\训练模型高温压缩-3.4"
shap_output_dir = r"D:\文成数据库\4.20HTC-SHAP分析"

# 创建SHAP输出目录
if not os.path.exists(shap_output_dir):
    os.makedirs(shap_output_dir)

# ====================== 2. 加载模型和数据 ======================
df = pd.read_excel(r"D:\文成数据库\Nb-Si数据库2.28-高温压缩.xlsx")

model_path = os.path.join(model_dir, 'htc_model.pth')
model_structure_path = os.path.join(model_dir, 'high_temperature_compression_model.joblib')
scaler_path = os.path.join(model_dir, 'final_scaler.joblib')
features_path = os.path.join(model_dir, 'final_features.txt')

# 读取特征列表
with open(features_path, 'r', encoding='utf-8') as f:
    final_features = [line.strip() for line in f.readlines()]

# 设置特征名称列表供SHAP可视化使用
feature_names_list = final_features

print("加载模型使用的特征:", final_features)

# 加载标准化器
scaler = joblib.load(scaler_path)

# 加载模型（两种方式尝试）
try:
    # 方式1：直接加载joblib保存的模型
    model_structure = joblib.load(model_structure_path)
    model = model_structure
    print("成功通过joblib直接加载模型")
except Exception as e:
    print(f"通过joblib加载模型失败，错误: {e}")
    print("尝试通过加载模型参数的方式重建模型...")
    # 方式2：创建新模型实例并加载参数
    model = MyModel(len(final_features))

try:
    model.load_state_dict(torch.load(model_path, map_location='cpu'))
    print("成功加载模型权重")
except Exception as e:
    print(f"加载模型权重出错: {e}")
    raise

device = torch.device("cpu")  # 使用CPU可能更稳定
model.to(device)
model.eval()

# ====================== 3. 为SHAP准备数据 ======================
# 提取特征数据并标准化
X_orig = df[final_features].values  # 原始未标准化的特征值
X_scaled = scaler.transform(X_orig)  # 标准化后的特征值
# 目标变量
y = df['high temperature compression'].values

# 转换为PyTorch张量
X_tensor = torch.tensor(X_scaled, dtype=torch.float32).to(device)

# ====================== 4. 创建自定义模型包装器 =====================
class ShapModelWrapper:
    def __init__(self, model):
        self.model = model
        
    def __call__(self, X):
        if isinstance(X, np.ndarray):
            X_tensor = torch.tensor(X, dtype=torch.float32).to(device)
        else:
            X_tensor = X
            
        with torch.no_grad():
            output = self.model(X_tensor)
            
        return output.cpu().numpy()

# ====================== 5. SHAP分析 - 使用Kernel解释器 ======================
print("创建SHAP解释器...")

# 创建背景数据集
max_bg_samples = min(100, X_scaled.shape[0])
bg_indices = np.random.choice(X_scaled.shape[0], max_bg_samples, replace=False)
bg_data = X_scaled[bg_indices]

print(f"使用所有 {X_scaled.shape[0]} 个样本进行SHAP分析")
sample_data = X_scaled
sample_indices = np.arange(X_scaled.shape[0])

model_wrapper = ShapModelWrapper(model)
explainer = shap.KernelExplainer(model_wrapper, bg_data)
print("成功创建SHAP KernelExplainer")

print("计算SHAP值...")
shap_values = explainer.shap_values(sample_data)

# 处理SHAP值形状
if isinstance(shap_values, list) and len(shap_values) == 1:
    shap_values = shap_values[0]

# 确保SHAP值是2D数组
if len(np.array(shap_values).shape) > 2:
    shap_values = np.squeeze(shap_values)
    
print(f"处理后SHAP值形状: {np.array(shap_values).shape}")

# ====================== 6. 计算力图数据（仅10个样本） ======================
print("计算力图数据...")

# 创建力图数据保存表格
force_plot_data = []

# 为力图创建专用的数据结构
force_samples = min(10, len(sample_data))  # 仅计算10个样本的力图

for i in range(force_samples):
    try:
        # 收集力图数据
        sample_force_data = {
            "样本索引": sample_indices[i],
            "基准值": explainer.expected_value
        }
        
        # 添加每个特征的SHAP值和原始特征值
        for feat_idx, feat_name in enumerate(feature_names_list):
            sample_force_data[f"{feat_name}_SHAP值"] = shap_values[i][feat_idx]
            sample_force_data[f"{feat_name}_特征值"] = X_orig[i][feat_idx]  # 使用原始特征值
        
        force_plot_data.append(sample_force_data)
        print(f"成功计算样本 {sample_indices[i]} 的力图数据")
        
    except Exception as e:
        print(f"处理样本 {i} 的力图数据时出错: {e}")

# 保存力图数据到Excel
if force_plot_data:
    force_plot_df = pd.DataFrame(force_plot_data)
    force_plot_df.to_excel(os.path.join(shap_output_dir, 'force_plot_data.xlsx'), index=False)
    print("力图数据已保存到Excel")

# ====================== 7. 特征重要性排名 ======================
print("计算特征重要性排名...")

# 计算平均绝对SHAP值作为特征重要性指标
mean_abs_shap = np.mean(np.abs(shap_values), axis=0)
importance_pairs = sorted(zip(feature_names_list, mean_abs_shap), key=lambda x: x[1], reverse=True)
total_importance = np.sum(mean_abs_shap)

print("\n特征重要性排名 (基于平均绝对SHAP值):")
for i, (feature, value) in enumerate(importance_pairs, 1):
    rel_importance = (value / total_importance) * 100
    print(f"{i}. {feature}: {value:.4f} ({rel_importance:.2f}%)")

# 将特征重要性保存到文件
importance_df = pd.DataFrame({
    'Feature': [p[0] for p in importance_pairs],
    'SHAP Value': [p[1] for p in importance_pairs],
    'Relative Importance (%)': [(p[1] / total_importance) * 100 for p in importance_pairs]
})
importance_df.to_excel(os.path.join(shap_output_dir, 'feature_importance.xlsx'), index=False)
print("特征重要性已保存到Excel文件")

# ====================== 8. 保存所有数据到Excel ======================
print("\n保存SHAP分析结果数据到Excel...")

# 获取预测值
with torch.no_grad():
    all_predictions = model(torch.tensor(X_scaled, dtype=torch.float32).to(device)).cpu().numpy().flatten()

# 合并SHAP值和原始特征值
all_data = pd.DataFrame()
for i, feat in enumerate(feature_names_list):
    all_data[f"{feat}_SHAP"] = shap_values[:, i]
    all_data[f"{feat}_原始值"] = X_orig[:, i]  # 使用原始特征值
all_data['预测值'] = all_predictions
all_data['样本索引'] = sample_indices

# 保存到Excel
all_data.to_excel(os.path.join(shap_output_dir, 'all_samples_shap_values.xlsx'), index=False)
print("所有样本的SHAP值和原始特征值已保存到Excel")

# ====================== 9. 保存热图数据 ======================
# 根据样本分类创建热图数据
try:
    # 基于预测值将样本分为高值和低值两组
    median_pred = np.median(all_predictions)
    high_indices = all_predictions > median_pred
    low_indices = ~high_indices
    
    # 高性能组
    high_group_data = pd.DataFrame()
    for i, feat in enumerate(feature_names_list):
        high_group_data[f"{feat}_SHAP"] = shap_values[high_indices, i]
        high_group_data[f"{feat}_原始值"] = X_orig[high_indices, i]  # 使用原始特征值
    high_group_data['预测值'] = all_predictions[high_indices]
    high_group_data['样本索引'] = sample_indices[high_indices]
    high_group_data.to_excel(os.path.join(shap_output_dir, 'high_performance_group_data.xlsx'), index=False)
    
    # 低性能组
    low_group_data = pd.DataFrame()
    for i, feat in enumerate(feature_names_list):
        low_group_data[f"{feat}_SHAP"] = shap_values[low_indices, i]
        low_group_data[f"{feat}_原始值"] = X_orig[low_indices, i]  # 使用原始特征值
    low_group_data['预测值'] = all_predictions[low_indices]
    low_group_data['样本索引'] = sample_indices[low_indices]
    low_group_data.to_excel(os.path.join(shap_output_dir, 'low_performance_group_data.xlsx'), index=False)
    
    print("热图数据（高性能和低性能组数据）已保存到Excel")

    # 保存特征统计摘要数据（热图分析结果）
    feature_summary_data = []
    for i, feat in enumerate(feature_names_list):
        feat_data = {
            '特征名称': feat,
            '平均SHAP值': np.mean(shap_values[:, i]),
            '平均绝对SHAP值': np.mean(np.abs(shap_values[:, i])),
            '最大正SHAP值': np.max(shap_values[:, i]),
            '最大负SHAP值': np.min(shap_values[:, i]),
            'SHAP值标准差': np.std(shap_values[:, i]),
            '高性能组平均SHAP值': np.mean(shap_values[high_indices, i]),
            '低性能组平均SHAP值': np.mean(shap_values[low_indices, i])
        }
        feature_summary_data.append(feat_data)

    feature_summary_df = pd.DataFrame(feature_summary_data)
    feature_summary_df.to_excel(os.path.join(shap_output_dir, 'feature_summary_statistics.xlsx'), index=False)
    print("特征统计摘要数据（热图分析结果）已保存到Excel")
    
except Exception as e:
    print(f"创建热图数据时出错: {e}")

print(f"\nSHAP分析完成! 所有数据已保存到: {shap_output_dir}")

加载模型使用的特征: ['VEC', 'am', 'Mean_热膨胀(1/k)', 'Mean_比热容J/g/k', 'Mean_Pyykko(Triple Bond) (pm)', 'Var_E13 Electron affinity']
成功通过joblib直接加载模型
成功加载模型权重
创建SHAP解释器...
使用所有 34 个样本进行SHAP分析
成功创建SHAP KernelExplainer
计算SHAP值...


  0%|          | 0/34 [00:00<?, ?it/s]

处理后SHAP值形状: (34, 6)
计算力图数据...
成功计算样本 0 的力图数据
成功计算样本 1 的力图数据
成功计算样本 2 的力图数据
成功计算样本 3 的力图数据
成功计算样本 4 的力图数据
成功计算样本 5 的力图数据
成功计算样本 6 的力图数据
成功计算样本 7 的力图数据
成功计算样本 8 的力图数据
成功计算样本 9 的力图数据
力图数据已保存到Excel
计算特征重要性排名...

特征重要性排名 (基于平均绝对SHAP值):
1. Mean_比热容J/g/k: 182.6925 (36.27%)
2. Mean_Pyykko(Triple Bond) (pm): 121.2739 (24.08%)
3. am: 101.6747 (20.18%)
4. Mean_热膨胀(1/k): 46.1518 (9.16%)
5. VEC: 28.0843 (5.58%)
6. Var_E13 Electron affinity: 23.8555 (4.74%)
特征重要性已保存到Excel文件

保存SHAP分析结果数据到Excel...
所有样本的SHAP值和原始特征值已保存到Excel
热图数据（高性能和低性能组数据）已保存到Excel
特征统计摘要数据（热图分析结果）已保存到Excel

SHAP分析完成! 所有数据已保存到: D:\文成数据库\4.20HTC-SHAP分析
